# Titanic — предсказание выживаемости

## Цель
Предсказать выжил ли пассажир на основе характеристик (возраст, пол, класс билета и др.)

## Данные
- Источник: Titanic dataset (891 пассажир, 12 колонок)
- Целевая переменная: `Survived` (0 = погиб, 1 = выжил)

## Стек
Python · pandas · scikit-learn · numpy

## импорты и загрузка данных

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Загрузка данных
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
print(f"Размер датасета: {df.shape}")
df.head()

Размер датасета: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Очистка данных

In [3]:
# Пропуски
print("Пропуски до очистки:")
print(df.isnull().sum())

# Заполняем пропуски
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df = df.drop(columns=['Cabin'])

# Кодируем категории
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

print("\nПропуски после очистки:")
print(df.isnull().sum())

Пропуски до очистки:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Пропуски после очистки:
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


## Подготовка данных и Baseline

In [4]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
X = df[features]
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Baseline — предсказываем самый частый класс
baseline = y_train.mode()[0]
baseline_pred = [baseline] * len(y_test)
baseline_acc = accuracy_score(y_test, baseline_pred)

print(f"Baseline accuracy: {baseline_acc:.2f}")

Baseline accuracy: 0.59


## Сравнение моделей

In [5]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=2000, solver="liblinear", random_state=42))
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

results = []
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    results.append({
        'Model': name,
        'Mean Accuracy': scores.mean().round(2),
        'Std': scores.std().round(2)
    })

results_df = pd.DataFrame(results).sort_values('Mean Accuracy', ascending=False)
print(results_df)

                 Model  Mean Accuracy   Std
2        Random Forest           0.81  0.04
0  Logistic Regression           0.78  0.02
1        Decision Tree           0.78  0.03


## Выводы

- Baseline (предсказывать самый частый класс): 0.59
- Лучшая модель: Random Forest — accuracy 0.81 (+22% к baseline)
- Logistic Regression и Decision Tree: 0.78
- Random Forest стабильнее — предсказания менее зависят от разбивки данных
- Самые важные признаки: Fare, Sex, Age (богатые, женщины и дети выживали чаще)

## Что можно улучшить
- Добавить feature engineering (например, размер семьи = SibSp + Parch)
- Подобрать гиперпараметры Random Forest
- Попробовать градиентный бустинг (следующий шаг после этой недели)